# An interactive critical-path Gantt: sample, CSV or Excel

Run this standalone notebook with its built-in example, a CSV, or the Excel template's input table. It uses the same scheduling statements as the desktop-checked v1.3 workbook, with Excel's cell-to-cell handoffs removed. Nothing is written back to the workbook. The built-in sample and supplied CSV use the same 23-task house-plan inputs as the XLSX, so results can be compared directly.

**For a quick result:** leave `INPUT_MODE = "sample"` and run all cells. No input files or repository checkout are needed. The results, warnings and interactive chart appear inside this notebook. File export is off by default; set `EXPORT_HTML = True` to save an HTML snapshot and sample CSV template. For your own data, choose `csv` or `xlsx` in step 1.

**For learning:** follow the numbered sections. A notebook is a document containing explanations and runnable Python cells. Variables carry forward into later cells; run them in order. If a cell fails, correct the input and restart/run all. Older outputs can remain visible after an error and must not be treated as a fresh schedule.

**Scope:** up to 100 consecutive tasks; elapsed weeks; FS, SS and FF relationships with lags; milestones and anchored hammocks. Progress is informational. Fixed starts override the forward pass; the backward pass ignores them, so positive float is not permission to miss a fixed date. Use at your own risk. This is a planning aid, not a professional scheduling system or DCMA assessment.

Unlike Python in Excel, this notebook does not require a Python-in-Excel licence. It runs in your selected Python environment: locally, or on Kaggle/Colab servers if you use those services. Uploading your own data there means cloud processing. The HTML export contains schedule information: share it with the same care as the inputs.

## 1. Choose sample, CSV or XLSX

Leave the defaults for an immediate demonstration. To use a CSV, set `INPUT_MODE = "csv"` and point `CSV_PATH` to the uploaded/saved file. To read the Excel template, set `INPUT_MODE = "xlsx"` and enter its folder/filename. Use `r"C:\..."` for Windows paths.

**Colab:** upload this notebook, connect a Python runtime, and choose **Runtime → Run all**. For CSV mode, upload the CSV using the Files sidebar and copy its path (typically `/content/your_plan.csv`).

**Kaggle:** import this notebook, use a Python CPU session, and run all cells. For CSV mode, attach your dataset and copy its actual `/kaggle/input/...` path. Keep outputs under `/kaggle/working` (the default working directory), not the read-only input folder. No GPU is needed.

NumPy, pandas and Plotly are normally included in these environments. The next cell prints installed versions and gives a clear message if a package is missing. Do not install this repository's Windows/Jupyter dependency lock into a hosted runtime. If needed, run `%pip install numpy pandas plotly` in a separate cell, then restart the session. Only XLSX mode needs openpyxl (`%pip install openpyxl` if absent). Nothing is installed automatically.

The chart always appears inside the notebook. `EXPORT_HTML = False` creates no output files. Set it to `True` to save a local HTML snapshot and sample CSV template in the output folder. Rerunning export replaces the same output filenames. Keep them separate from your source inputs.

In [ ]:
from pathlib import Path

WORKBOOK_FOLDER = Path(".")
INPUT_MODE = "sample"  # "sample", "csv", or "xlsx"
CSV_PATH = Path("sample_schedule.csv")  # Replace with your own file's path.
WORKBOOK_FILENAME = "CritPath_GanttChart_Template_v1.3.xlsx"
OUTPUT_FOLDER = Path("outputs")
OUTPUT_FILENAME = "critical_path_gantt.html"

workbook_path = (WORKBOOK_FOLDER / WORKBOOK_FILENAME).expanduser().resolve()
output_path = (OUTPUT_FOLDER / OUTPUT_FILENAME).expanduser().resolve()
if output_path == workbook_path or output_path.suffix.lower() != ".html":
    raise ValueError("Choose a separate .html output file.")
if output_path == CSV_PATH.expanduser().resolve():
    raise ValueError("Choose a separate output file; do not overwrite your CSV.")

## 2. Sample table and input readers

The table below is ordinary editable Python data copied from the supplied XLSX house plan, including its descriptive columns. Each row is a task; durations/lags use elapsed weeks. It demonstrates FS, SS, FF, milestones, parallel work and hammocks. It can run by itself in a copied notebook; the workbook is not read in sample mode.

**CSV schema:** required headers are `TaskID`, `TaskType`, `Duration`, `Predecessors`, `Constraints` (case-sensitive, in any order). `TaskTitle` is recommended; if omitted it defaults to the ID. Other descriptive columns are optional and retained. Keep IDs as text. Blank constraint cells mean no override; at least one date is required. Prefer ISO dates (`2026-10-20`). Quote predecessor lists containing commas, e.g. `"0030 FS, 0040 FS"`. Do not leave blank rows between tasks. The final cell also exports this sample as a CSV template.

`openpyxl` reads the template's `Workplan` / `Table1` input area. It does not run the workbook's Python cells, trust their saved results, or execute code supplied by another workbook. The scheduling code below is the reviewed v1.3 code embedded in this notebook.

Text TaskIDs such as `0030` remain text. Blanks between tasks remain present so validation can report a gap. Inputs must be literal values: the reader stops if it encounters an Excel formula in A:M. Use real Excel dates or an unambiguous text date such as `2026-10-20`. At least one constraint date is needed to place the plan on the calendar.

The readers below are included in full for transparency. You normally do not need to edit them.

In [ ]:
from importlib.metadata import version, PackageNotFoundError
for package in ("numpy", "pandas", "plotly"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        raise ImportError(f"Missing {package}. Install it in this notebook's environment, restart, and run all.")
import numpy as np
import pandas as pd
from IPython.display import display, FileLink
"""Read only input values; never evaluate workbook Python or save the workbook."""
from pathlib import Path
import warnings
import pandas as pd


def prepare_input(frame):
    """Check the shared schema; keep IDs and blank rows for core validation."""
    required = ['TaskID', 'TaskType', 'Duration', 'Predecessors', 'Constraints']
    missing = [name for name in required if name not in frame.columns]
    if missing or frame.columns.duplicated().any():
        raise ValueError('Missing or duplicate headers. Required: ' + ', '.join(required))
    frame = frame.copy()
    frame = frame.mask(frame.eq(''), float('nan'))
    # Parse ISO text explicitly before the unchanged day-first Excel parser.
    # Otherwise dates such as 2026-10-01 could be read as January 10.
    import re
    def iso_date(value):
        if isinstance(value, str) and re.fullmatch(r'\d{4}-\d{2}-\d{2}', value.strip()):
            try:
                return pd.Timestamp(value.strip())
            except ValueError:
                pass  # Leave invalid text for the scheduler's clear error.
        return value
    frame['Constraints'] = frame['Constraints'].map(iso_date)
    if 'TaskTitle' not in frame:
        frame['TaskTitle'] = frame['TaskID']
    return frame


def read_schedule_csv(path):
    """Read CSV as text: preserve 0030/NA IDs and report malformed rows."""
    import csv
    path = Path(path).expanduser().resolve()
    with path.open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.reader(stream)
        headers = next(reader, [])
        if len(headers) != len(set(headers)):
            raise ValueError('Duplicate CSV column headers.')
        rows = []
        for number, row in enumerate(reader, 2):
            if not row:
                row = [''] * len(headers)  # Preserve gaps for scheduler checks.
            if len(row) != len(headers):
                raise ValueError(f'CSV record {number} has {len(row)} fields; expected {len(headers)}. '
                                 'Quote predecessor lists containing commas.')
            rows.append(row)
    return prepare_input(pd.DataFrame(rows, columns=headers))


def read_workplan(path):
    """Read the v1.2 Workplan table, preserving text IDs and empty interior rows."""
    # Optional dependency: sample and CSV modes do not need openpyxl.
    from openpyxl import load_workbook
    from openpyxl.utils.cell import range_boundaries
    path = Path(path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f'Workbook not found: {path}. Check the folder and filename above.')
    if path.suffix.lower() != '.xlsx':
        raise ValueError('Choose an .xlsx workbook.')
    # openpyxl cannot evaluate =PY(). That is intentional: only inputs are read.
    # It warns about Excel extensions that it would discard on saving. We never
    # save this object, so the original extensions and workbook remain intact.
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='.*extension is not supported and will be removed')
        book = load_workbook(path, data_only=False, keep_links=False)
    try:
        if 'Workplan' not in book.sheetnames:
            raise ValueError('The workbook needs a Workplan sheet.')
        sheet = book['Workplan']
        if 'Table1' not in sheet.tables:
            raise ValueError('Workplan needs the template input table named Table1.')
        first_col, header_row, last_col, last_row = range_boundaries(sheet.tables['Table1'].ref)
        if (first_col, header_row, last_col) != (1, 2, 13):
            raise ValueError('Expected the template input table in A2:M...; keep its original layout.')
        headers = [sheet.cell(header_row, c).value for c in range(first_col, last_col + 1)]
        required = {'TaskID', 'TaskType', 'Duration', 'Predecessors', 'Constraints'}
        if len(set(headers)) != len(headers) or not required.issubset(headers):
            raise ValueError('Missing or duplicate input headers; required: ' + ', '.join(sorted(required)))
        # Include populated rows beyond the table so they cannot silently vanish.
        # The unchanged scheduler enforces its 100-task limit and gap rules.
        values = []
        for row in sheet.iter_rows(min_row=header_row + 1, max_row=max(last_row, sheet.max_row),
                                   min_col=first_col, max_col=last_col):
            record = []
            for cell in row:
                if cell.data_type == 'f':
                    raise ValueError(f'Formula input in {cell.coordinate}: enter a literal value in a working copy. '
                                     'The notebook does not calculate Excel input formulas.')
                value = cell.value
                record.append(None if isinstance(value, str) and not value.strip() else value)
            values.append(record)
        # Keep blanks between tasks; trim only unused trailing rows.
        while values and all(value is None for value in values[-1]):
            values.pop()
        frame = pd.DataFrame(values, columns=headers)
        # Excel blanks arrive as None; pandas' date parser expects missing values.
        frame = frame.where(frame.notna(), float('nan'))
        return prepare_input(frame)
    finally:
        book.close()


In [ ]:
# Same house-plan inputs as the supplied XLSX; editable without Excel.
SAMPLE_COLUMNS = ['TaskID', 'TaskType', 'Stage', 'TaskTitle', 'TaskDescription', 'Task Grouping A', 'Task Grouping B', 'Task Grouping C', 'Task Grouping D', 'Duration', 'Progress', 'Predecessors', 'Constraints']
SAMPLE_ROWS = [
    ['0010', 'Summary', '', 'Build a house', '', '', '', '', '', 0, '', '0020 SS,0230 FF', ''],
    ['0020', 'Milestone', '', 'Start Build', '', '', '', '', '', 0, '', '', '2026-10-20'],
    ['0030', 'Activity', '', ' PREPARE SITE', '', '', '', '', '', 4, '', '0020 FS', ''],
    ['0040', 'Activity', '', ' ERECT HOARDING', '', '', '', '', '', 2, '', '0030 FS', ''],
    ['0050', 'Activity', '', ' EXCAVATE FOUNDATION', '', '', '', '', '', 7, '', '0030 FS', ''],
    ['0060', 'Activity', '', ' LAY DRAINS', '', '', '', '', '', 4, '', '0040 FS', ''],
    ['0070', 'Activity', '', ' CONCRETE FOUNDATION', '', '', '', '', '', 5, '', '0050 FS,0060 FS', ''],
    ['0080', 'Activity', '', ' BRICKWORK', '', '', '', '', '', 9, '', '0070 FS', ''],
    ['0090', 'Activity', '', ' INSTALL WINDOWS', '', '', '', '', '', 6, '', '0080 SS+2', ''],
    ['0100', 'Activity', '', ' ROOF TRUSSES', '', '', '', '', '', 6, '', '0080 FS,0090 FS', ''],
    ['0110', 'Activity', '', ' ROUGH PLUMBING & ELECTRIC', '', '', '', '', '', 4, '', '0100 SS+2', ''],
    ['0120', 'Activity', '', ' ROOF FINISH', '', '', '', '', '', 7, '', '0100 FS', ''],
    ['0130', 'Activity', '', ' LANDSCAPING', '', '', '', '', '', 4, '', '0150 SS', ''],
    ['0140', 'Summary', '', ' PLASTER WALLS', '', '', '', '', '', 0, '', '0150 SS,0170 FF', ''],
    ['0150', 'Activity', '', 'WALL A', '', '', '', '', '', 2, '', '0110 FS,0120 FS', ''],
    ['0160', 'Activity', '', 'WALL B', '', '', '', '', '', 3, '', '0150 FS+3', ''],
    ['0170', 'Activity', '', 'WALL C', '', '', '', '', '', 2, '', '0160 FS+2', ''],
    ['0180', 'Activity', '', ' FINISH ELECTRICS', '', '', '', '', '', 4, '', '0170 FS', ''],
    ['0190', 'Activity', '', ' FINISH PLUMBING', '', '', '', '', '', 4, '', '0170 FS', ''],
    ['0200', 'Activity', '', ' DECORATE', '', '', '', '', '', 4, '', '0180 FS, 0190 FS', ''],
    ['0210', 'Activity', '', ' CLEAN DOWN', '', '', '', '', '', 1, '', '0200 FS', ''],
    ['0220', 'Activity', '', ' SNAGGING', '', '', '', '', '', 1, '', '0130 FS ,0210 FS', ''],
    ['0230', 'Milestone', '', 'Build Complete', '', '', '', '', '', 0, '', '0220 FS', ''],
]
sample_data = pd.DataFrame(SAMPLE_ROWS, columns=SAMPLE_COLUMNS)

if INPUT_MODE == "sample":
    input_data = prepare_input(sample_data)
    source_name = "Built-in house-plan sample (same inputs as XLSX)"
elif INPUT_MODE == "csv":
    input_data = read_schedule_csv(CSV_PATH)
    source_name = CSV_PATH.name
elif INPUT_MODE == "xlsx":
    input_data = read_workplan(workbook_path)
    source_name = workbook_path.name
else:
    raise ValueError('INPUT_MODE must be "sample", "csv", or "xlsx".')
print(f"Read {len(input_data)} input rows: {source_name}")
display(input_data)

## 3. Validate task rows and durations

`d` is our working copy of the input table. The checks reject missing/duplicate IDs, gaps, invalid types, and negative or nonnumeric durations. Blank durations keep the workbook's existing zero-duration convention. `T` preserves task order; dictionaries let later calculations look up each task by ID. Milestones have zero duration, and hammock durations are calculated later.

In [ ]:
import re

d=input_data.copy(deep=True)
# Outputs spill consecutively from row 3, so task inputs must be consecutive.
present=d["TaskID"].notna()
populated=d.astype(object).where(d.notna(),"").astype(str).apply(lambda c:c.str.strip().ne("")).any(axis=1)
missing=populated & ~present
if missing.any():
    raise ValueError("Missing TaskID on input row(s): "+", ".join(str(i+3) for i in np.flatnonzero(missing.to_numpy())))
positions=np.flatnonzero(present.to_numpy())
if len(positions) and not present.iloc[:positions[-1]+1].all():
    raise ValueError("Remove blank task-row gaps: enter tasks consecutively from row 3")
if len(positions)>100:
    raise ValueError("This template displays at most 100 task rows; shorten the schedule or use a larger scheduling tool")
d=d.dropna(subset=["TaskID"]).copy()

if d.empty:
    raise ValueError("Enter at least one task")
d["TaskID"]=d["TaskID"].astype(str).str.strip()
if d["TaskID"].eq("").any():
    raise ValueError("TaskID cannot be whitespace")
d["TaskType"]=d["TaskType"].fillna("Activity").astype(str).str.strip().str.capitalize()
raw_duration=d["Duration"]
number_duration=pd.to_numeric(raw_duration,errors="coerce")
nonblank=raw_duration.notna() & raw_duration.astype(str).str.strip().ne("")
bad_duration=nonblank & (number_duration.isna() | ~np.isfinite(number_duration) | number_duration.lt(0))
if bad_duration.any():
    raise ValueError("Duration must be a finite nonnegative number for: "+", ".join(d.loc[bad_duration,"TaskID"]))
d["Duration"]=number_duration.fillna(0)
d["Predecessors"]=d["Predecessors"].fillna("").astype(str).str.strip()

T=d["TaskID"].tolist()
typ=dict(zip(T,d["TaskType"]))
rawdur=dict(zip(T,d["Duration"]))

valid_types={"Activity","Milestone","Summary"}
bad_types=[t for t in T if typ[t] not in valid_types]

if bad_types:
    raise ValueError("Invalid TaskType for: "+", ".join(bad_types))

if len(T)!=len(set(T)):
    raise ValueError("Duplicate TaskID found")

dur={
    t:0.0 if typ[t] in ("Milestone","Summary") else float(rawdur[t])
    for t in T
}

## 4. Establish the date origin and fixed starts

The earliest constraint date becomes week zero (`origin`). Other dates become elapsed-week offsets in `fix`. These are fixed **start overrides**, not a full constraint engine. Dates are normalised to midnight; time-of-day input is not supported. Text dates use the workbook's day-first parser; prefer real Excel dates or ISO text to avoid ambiguity.

In [ ]:
dates=pd.to_datetime(d["Constraints"],errors="coerce",dayfirst=True)
nonblank_date=d["Constraints"].notna() & d["Constraints"].astype(str).str.strip().ne("")
bad_date=nonblank_date & dates.isna()
if bad_date.any():
    raise ValueError("Invalid Constraint date for: "+", ".join(d.loc[bad_date,"TaskID"]))
valid=dates.dropna()

if valid.empty:
    raise ValueError("At least one Constraint date is required")

origin=valid.min().normalize()

fix={}

for i,r in d.iterrows():
    x=dates.loc[i]
    fix[r["TaskID"]]=(
        None
        if pd.isna(x)
        else (x.normalize()-origin).days/7
    )

## 5. Connect predecessors and check for cycles

Each predecessor becomes `(TaskID, relationship, lag)`. `pred` looks backward; `succ` looks forward. Lag is measured in elapsed weeks and can be negative. A cycle means tasks ultimately depend on themselves, so calculation stops. Hammocks use SS/FF anchors and cannot drive ordinary tasks. The compatibility parser is retained exactly as in Excel; the documented relationships remain FS, SS and FF.

In [ ]:
pred={t:[] for t in T}
succ={t:[] for t in T}

pat=re.compile(
    r"^\s*(\S+)\s+(FS|SS|FF|SF)"
    r"(?:\s*([+-])\s*(\d+(?:\.\d+)?))?\s*$",
    re.I
)

for r in d.itertuples():
    if not r.Predecessors:
        continue

    for text in r.Predecessors.split(","):
        text=text.strip()
        m=pat.match(text)

        if not m:
            raise ValueError("Invalid predecessor: "+text)

        p,rel,sg,n=m.groups()
        rel=rel.upper()

        if p not in T:
            raise ValueError("Unknown predecessor: "+p)

        if typ[p]=="Summary":
            raise ValueError(r.TaskID+" cannot depend on Summary task "+p)

        if p==r.TaskID:
            raise ValueError("Task cannot depend on itself: "+p)

        lag=float(n or 0)

        if sg=="-":
            lag=-lag

        pred[r.TaskID].append((p,rel,lag))

        if typ[r.TaskID]!="Summary":
            succ[p].append((r.TaskID,rel,lag))

for t in T:
    if typ[t]=="Summary":
        invalid=[rel for p,rel,lag in pred[t] if rel not in ("SS","FF")]

        if invalid:
            raise ValueError(
                t+" Summary only supports SS and FF anchors"
            )

# Reject all dependency cycles, including zero-lag SS cycles.
degree={t:len(pred[t]) for t in T}
following={t:[] for t in T}
for t in T:
    for p,rel,lag in pred[t]:
        following[p].append(t)
ready=[t for t in T if degree[t]==0]
for p in ready:
    for t in following[p]:
        degree[t]-=1
        if degree[t]==0:
            ready.append(t)
if len(ready)!=len(T):
    raise ValueError("Dependency cycle detected; check: "+", ".join(t for t in T if degree[t]>0))
if all(typ[t]=="Summary" for t in T):
    raise ValueError("Enter at least one Activity or Milestone")

## 6. Forward pass: find early starts and finishes

`ES` and `EF` are early start/finish offsets in weeks. Each relationship imposes a lower bound on a task's dates. Repeated passes propagate changes even if rows are not in dependency order. This deliberately retains the workbook's existing approach. A fixed start then overrides the dependency-derived start. The iteration limit protects against nonconvergence.

In [ ]:
REG=[t for t in T if typ[t]!="Summary"]
SUM=[t for t in T if typ[t]=="Summary"]

ES={t:0.0 for t in T}
EF={t:dur[t] for t in T}
CD=dur.copy()

lim=max(10,len(T)*4)

for _ in range(lim):
    ch=False

    for t in REG:
        s=0.0

        for p,r,l in pred[t]:
            if typ[p]=="Summary":
                raise ValueError(
                    t+" cannot depend on Summary task "+p
                )

            if r=="FS":
                x=EF[p]+l
            elif r=="SS":
                x=ES[p]+l
            elif r=="FF":
                x=EF[p]+l-dur[t]
            else:
                x=ES[p]+l-dur[t]

            s=max(s,x)

        if fix[t] is not None:
            s=fix[t]

        f=s+dur[t]

        if s!=ES[t] or f!=EF[t]:
            ES[t],EF[t]=s,f
            ch=True

    if not ch:
        break
else:
    raise ValueError(
        "Forward pass did not converge; check dependency cycles"
    )

## 7. Make fixed-start conflicts visible

A fixed start can contradict a predecessor. The workbook retains the override and creates a warning; so does this notebook. `WARN` collects messages per task. Warnings do not repair the schedule and should be resolved before you rely on it. Invalid inputs and cycles are blocking errors instead.

In [ ]:
# Keep fixed-start overrides, but expose violated ties in worksheet output.
WARN={t:[] for t in T}
for t in REG:
    for p,r,l in pred[t]:
        left,right=(ES[t],EF[p]+l) if r=="FS" else (ES[t],ES[p]+l) if r=="SS" else (EF[t],EF[p]+l) if r=="FF" else (EF[t],ES[p]+l)
        if left+1e-9<right:
            WARN[t].append(t+": fixed start conflicts with "+p+" "+r+" (shortfall "+str(round(right-left,4))+" weeks)")

## 8. Backward pass: find late dates

`PF` is the latest early finish among activities/milestones. Starting from that project finish, the backward pass derives late finishes (`LF`) and late starts (`LS`) from successor relationships. **It does not pin fixed constraint dates.** This is why the float below cannot tell you whether a fixed date may be missed.

In [ ]:
PF=max(EF[t] for t in REG)

LF={t:PF for t in T}
LS={t:PF-dur[t] for t in T}

for _ in range(lim):
    ch=False

    for t in reversed(REG):
        f=PF

        for s,r,l in succ[t]:
            if r=="FS":
                x=LS[s]-l
            elif r=="SS":
                x=LS[s]-l+dur[t]
            elif r=="FF":
                x=LF[s]-l
            else:
                x=LF[s]-l+dur[t]

            f=min(f,x)

        s=f-dur[t]

        if s!=LS[t] or f!=LF[t]:
            LS[t],LF[t]=s,f
            ch=True

    if not ch:
        break
else:
    raise ValueError("Backward pass did not converge; check schedule inputs")

## 9. Draw hammock spans

A Summary here is a hammock: its earliest SS anchor sets its start, and its latest FF anchor sets its finish. It is a display span, not a hierarchical summary that automatically rolls up child tasks. It needs both anchors and cannot have a finish before its start.

In [ ]:
for t in SUM:
    ss=[
        ES[p]+l
        for p,r,l in pred[t]
        if r=="SS"
    ]

    ff=[
        EF[p]+l
        for p,r,l in pred[t]
        if r=="FF"
    ]

    if not ss:
        raise ValueError(
            t+" Summary requires at least one SS anchor"
        )

    if not ff:
        raise ValueError(
            t+" Summary requires at least one FF anchor"
        )

    ES[t]=min(ss)
    EF[t]=max(ff)

    if EF[t]<ES[t]:
        raise ValueError(
            t+" Summary finish is before its start"
        )

    CD[t]=EF[t]-ES[t]
    LS[t]=ES[t]
    LF[t]=EF[t]

## 10. Calculate finish float and calendar dates

Total finish float is `LF − EF`, rounded to four decimal places. Activities **and milestones** with float at or below 0.001 weeks are flagged critical, including negative float. Hammocks have no float/critical flag. `pd.Timedelta(weeks=...)` converts offsets into calendar dates using seven-day weeks, with no holidays or working-time calendar.

For fixed-duration activities and milestones, total float also equals `LS - ES`. `LateStart` and `LateFinish` expose those offsets in weeks. `FreeFloat` is the smallest slack across outgoing FS/SS/FF (and compatibility SF) endpoints, after subtracting lag, capped by `PF - EF` so this task does not extend the current project finish. A terminal task uses that finish bound alone. Negative values remain visible for conflicting inputs. Summary/hammock late dates and float are blank: they are display spans, not schedulable work. All float measures retain the fixed-start limitation described above.

In [ ]:
TFF={
    t:"" if typ[t]=="Summary" else round(LF[t]-EF[t],4)
    for t in T
}

CRIT={
    t:False
    if typ[t]=="Summary"
    else TFF[t]<=0.001
    for t in T
}

START={
    t:origin+pd.Timedelta(weeks=ES[t])
    for t in T
}

FINISH={
    t:origin+pd.Timedelta(weeks=EF[t])
    for t in T
}

# Free float: move this task without moving successor early dates.
# The current project finish is an implicit bound for every ordinary task.
# Keep negative slack visible when fixed starts contradict relationships.
FREE={}
for t in T:
    if typ[t]=="Summary":
        FREE[t]=""
        continue
    slack=[PF-EF[t]]
    for s,r,l in succ[t]:
        x=(ES[s]-EF[t]-l if r=="FS" else
           ES[s]-ES[t]-l if r=="SS" else
           EF[s]-EF[t]-l if r=="FF" else
           EF[s]-ES[t]-l)
        slack.append(x)
    FREE[t]=round(min(slack),4)

## 11. Assemble the schedule table

These ten output columns are the same ones returned to Excel. The next cell adds the original input fields and warnings for context. It does not change the calculation.

In [ ]:
schedule_values = pd.DataFrame({
    "CalcDuration":[round(CD[t],4) for t in T],
    "Start":[START[t] for t in T],
    "Finish":[FINISH[t] for t in T],
    "EarlyStart":[round(ES[t],4) for t in T],
    "EarlyFinish":[round(EF[t],4) for t in T],
        "LateStart":["" if typ[t]=="Summary" else round(LS[t],4) for t in T],
    "LateFinish":["" if typ[t]=="Summary" else round(LF[t],4) for t in T],
    "FreeFloat":[FREE[t] for t in T],
    "TotalFinishFloat":[TFF[t] for t in T],
    "IsCritical":[CRIT[t] for t in T]
})

In [ ]:
schedule = pd.concat([d.reset_index(drop=True), schedule_values], axis=1)
schedule["ScheduleWarning"] = ["; ".join(WARN[t]) for t in T]
display(schedule[["TaskID", "TaskType", "TaskTitle", "CalcDuration", "Start", "Finish",
                  "EarlyStart", "EarlyFinish", "LateStart", "LateFinish",
                  "FreeFloat", "TotalFinishFloat", "IsCritical", "ScheduleWarning"]])
warnings_table = schedule.loc[schedule.ScheduleWarning.ne(""), ["TaskID", "ScheduleWarning"]]
if warnings_table.empty:
    print("No fixed-start conflicts detected. Float still ignores fixed dates.")
else:
    print("WARNING: resolve the following fixed-start conflicts before relying on this plan.")
    display(warnings_table)

## 12. Explore the interactive Gantt

Red marks activities/milestones with zero or negative finish float; blue marks the others. Diamonds are milestones. Thin grey bars are hammocks. Hover for dates, float and warnings. Drag to pan; use the toolbar to zoom/reset. Click a legend entry to hide/show a category. Dependency connectors start hidden to keep the chart readable; click **Dependencies (toggle)** to reveal them. They connect the relationship endpoints; they are not a second scheduling calculation.

The chart has a continuous date axis, while the **model still calculates elapsed weeks**. It does not add daily working calendars, progress scheduling, or more sophisticated constraints. Large networks can be visually crowded. The chart code below controls presentation only.

In [ ]:
"""Presentation only: scheduling is performed by the workbook's copied logic."""
from html import escape
import pandas as pd
import plotly.graph_objects as go


def make_gantt(schedule, predecessors):
    """Return a local interactive chart; no web service or server is needed."""
    fig = go.Figure()
    positions = {row.TaskID: i for i, row in schedule.iterrows()}
    by_id = schedule.set_index('TaskID')
    # Connect the actual relationship endpoints. Lags are described on hover;
    # they are already included in scheduling, not recalculated by the chart.
    link_x, link_y, link_text = [], [], []
    for _, row in schedule.iterrows():
        if row.TaskType == 'Summary':
            continue  # Hammock anchors are display spans, not driving links.
        for previous, relation, lag in predecessors[row.TaskID]:
            start = by_id.loc[previous, 'Finish' if relation[0] == 'F' else 'Start']
            finish = row.Finish if relation[1] == 'F' else row.Start
            label = escape(f'{previous} → {row.TaskID}: {relation}, lag {lag:g} weeks')
            middle = start + (finish - start) / 2
            link_x.extend([start, middle, middle, finish, None])
            link_y.extend([positions[previous], positions[previous], positions[row.TaskID], positions[row.TaskID], None])
            link_text.extend([label] * 4 + [''])
    if link_x:
        fig.add_trace(go.Scatter(x=link_x, y=link_y, mode='lines', name='Dependencies (toggle)',
                                line=dict(color='#94a3b8', width=1), text=link_text,
                                hovertemplate='%{text}<extra></extra>', visible='legendonly'))
    kinds = [('Critical activity', '#dc3545', 'Activity', True),
             ('Other activity', '#2878b5', 'Activity', False),
             ('Summary / hammock', '#64748b', 'Summary', None),
             ('Critical milestone', '#dc3545', 'Milestone', True),
             ('Other milestone', '#2878b5', 'Milestone', False)]
    for label, color, task_type, critical in kinds:
        subset = schedule[schedule.TaskType.eq(task_type)]
        if critical is not None:
            subset = subset[subset.IsCritical.eq(critical)]
        if subset.empty:
            continue
        hover = []
        for _, row in subset.iterrows():
            title = row.get('TaskTitle', '')
            title = '' if pd.isna(title) else str(title)
            float_text = 'not applicable' if row.TaskType == 'Summary' else f'{row.TotalFinishFloat} weeks'
            hover.append(escape(f'{row.TaskID} — {title}') + '<br>' +
                         f'{row.Start:%d %b %Y %H:%M} → {row.Finish:%d %b %Y %H:%M}<br>' +
                         f'Duration: {row.CalcDuration:g} elapsed weeks<br>Finish float: {float_text}<br>' +
                         escape(row.ScheduleWarning or 'No fixed-start conflict detected'))
        y = [positions[t] for t in subset.TaskID]
        if task_type == 'Milestone':
            fig.add_trace(go.Scatter(x=subset.Start, y=y, mode='markers', name=label,
                                    marker=dict(symbol='diamond', size=12, color=color),
                                    text=hover, hovertemplate='%{text}<extra></extra>'))
        else:
            fig.add_trace(go.Bar(x=(subset.Finish - subset.Start).dt.total_seconds() * 1000,
                                base=list(subset.Start), y=y, orientation='h', name=label,
                                marker_color=color, width=.28 if task_type == 'Summary' else .62,
                                text=hover, textposition='none', hovertemplate='%{text}<extra></extra>'))
            # A zero-duration Activity still needs a visible marker.
            zero = subset.Start.eq(subset.Finish)
            if zero.any():
                fig.add_trace(go.Scatter(x=subset.loc[zero, 'Start'],
                                        y=[positions[t] for t in subset.loc[zero, 'TaskID']],
                                        mode='markers', marker=dict(size=8, color=color),
                                        name=label + ' (zero duration)', showlegend=False,
                                        text=[h for h, keep in zip(hover, zero) if keep],
                                        hovertemplate='%{text}<extra></extra>'))
    labels = []
    for _, row in schedule.iterrows():
        title = row.get('TaskTitle', '')
        title = '' if pd.isna(title) else str(title)
        labels.append(escape(f'{row.TaskID} · {title[:55]}'))
    fig.update_layout(template='plotly_white', title='Critical-path schedule · elapsed weeks',
                      barmode='overlay', height=max(520, len(schedule) * 27 + 210),
                      margin=dict(l=310, r=40, t=125, b=90),
                      legend=dict(orientation='h', y=1.07, x=0),
                      hoverlabel=dict(align='left'), dragmode='pan')
    fig.update_xaxes(type='date', title='Calendar dates · seven-day elapsed weeks',
                     showgrid=True, rangeslider_visible=False)
    fig.update_yaxes(tickmode='array', tickvals=list(range(len(schedule))), ticktext=labels,
                     range=[len(schedule) - .5, -.5], autorange=False, fixedrange=True, title=None)
    return fig


def export_gantt(fig, path, source_name, has_warnings):
    """Embed Plotly itself, so the saved chart has no CDN dependency."""
    from pathlib import Path
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    chart = fig.to_html(full_html=False, include_plotlyjs=True, config={'displaylogo': False, 'scrollZoom': True})
    status = 'Fixed-start conflicts are present. Review the notebook warnings.' if has_warnings else 'No fixed-start conflicts detected.'
    html = ('<!doctype html><html lang="en"><meta charset="utf-8"><title>Critical-path Gantt</title>'
            '<style>body{font-family:Arial,sans-serif;margin:24px;color:#1e293b}p{max-width:1100px}</style>'
            '<h1>Critical-path Gantt</h1><p>Source: ' + escape(source_name) + '</p><p>' + status +
            ' Float ignores fixed dates. Use at your own risk.</p><p>Hover for details; drag to pan; '
            'use the toolbar to zoom/reset. Click the dependency legend to show links. '
            'This is a snapshot: save input edits and rerun the notebook to update it.</p>' + chart + '</html>')
    path.write_text(html, encoding='utf-8')
    return path.resolve()


In [ ]:
fig = make_gantt(schedule, pred)
# Plotly detects local Jupyter, Colab and Kaggle renderers automatically.
fig.show(config={"displaylogo": False, "scrollZoom": True})

## 13. Optional: save an offline chart

Export is off by default. To save the chart you just generated, change `EXPORT_HTML` to `True` in the next cell, run that cell, then run the export cell below it. You do not need to rerun the scheduling calculation just to export its current result. After editing schedule inputs, rerun the calculation/chart first. The inline chart above appears with either setting. Turning export off does not delete files from an earlier run.

The HTML includes Plotly itself, so opening it needs no CDN or hosted application. It contains your schedule details. It is a **snapshot**, not a live input connection or editable planning application. After changing the sample/CSV/workbook inputs, save them and restart/run all notebook cells again. The HTML hover/zoom controls remain interactive; the schedule does not recalculate in the browser. In Colab/Kaggle, download output files from the Files/Output sidebar.

In [ ]:
EXPORT_HTML = False  # True saves HTML + sample CSV; inline chart always appears.

In [ ]:
if EXPORT_HTML:
    saved_chart = export_gantt(fig, output_path, source_name, not warnings_table.empty)
    print(f"Saved: {saved_chart}")
    sample_csv_path = output_path.parent / "sample_schedule.csv"
    if sample_csv_path not in {workbook_path, CSV_PATH.expanduser().resolve(), output_path}:
        sample_data.to_csv(sample_csv_path, index=False)
        print(f"CSV template: {sample_csv_path}")
# Relative links work with local Jupyter; hosted services also expose output
# files in their Files/Output sidebar, where they can be downloaded.
    import os
    display(FileLink(os.path.relpath(saved_chart, Path.cwd())))
else:
    print("Export is off. View the interactive chart above; set EXPORT_HTML = True to save files.")

## Where to go next

Try a duration change in the sample rows, your CSV, or a **working copy** of the workbook, then rerun all cells. Compare the finish date and critical flags. Preserve the golden examples. If you need actuals, resource levelling, custom calendars, or large schedules, use a suitable professional tool.

The inherited house example is adapted from Oracle Primavera Risk Analysis's *Example House Construct Plan*. See the repository README for attribution and the MIT licence for the project code.

**Traceability:** sections 3–5 correspond to Excel O2; 6–10 to P2; the ten-column table to Q2. Warning messages are the same `WARN` values displayed by CL2. Only Excel reads/tuple handoffs are replaced. The release was checked against the workbook scripts; the development tests are maintained separately.

**Libraries:** [openpyxl](https://openpyxl.readthedocs.io/en/stable/) reads inputs; [pandas](https://pandas.pydata.org/docs/) and [NumPy](https://numpy.org/doc/) support the existing calculation; [Plotly HTML export](https://plotly.com/python/interactive-html-export/) supplies the local chart. No NetworkX, Dash server or external scheduling engine is used.